# NjengaData — 01 Data Cleaning
**Housing Cost Transparency for the Kenyan Builder**

This notebook loads four verified data sources, cleans them,
and writes them into `njenga.db` — the single database that
powers all analysis and the affordability calculator.

| Source | File | Answers |
|---|---|---|
| CAHF HDCB Kenya 2022 | `data/processed/cahf_cost_breakdown.csv` | Where does James's money go? |
| KNBS CIPI 2021–2025 | `data/processed/knbs_material_index.csv` | Are costs rising? |
| KCHS Kenya 2022 | `data/processed/county_income.csv` | Can James afford to build now? |
| Integrum Regional 2021–2025 | `data/processed/integrum_regional_costs.csv` | Is it cheaper elsewhere? |

**Base period:** KNBS indices use December 2019 = 100  
**Geography:** CAHF is Nairobi-only. KNBS is national. KCHS covers 26 counties. Integrum covers 3 regions.


In [2]:
import os
from pathlib import Path

# Set this to your njenga-data repo root
repo_root = Path.home() / 'Documents' / 'njenga-data'

# If that path is wrong, update the line above to match yours
# e.g. Path('/home/miringu/Documents/njenga-data')

os.chdir(repo_root)
print(f'Working directory: {Path.cwd()}')
print(f'data/processed exists: {(repo_root / "data" / "processed").exists()}')
print(f'cahf CSV exists: {(repo_root / "data" / "processed" / "cahf_cost_breakdown.csv").exists()}')

Working directory: /home/miringu/Documents/njenga-data
data/processed exists: True
cahf CSV exists: True


## 0. Imports and Setup

In [3]:
import pandas as pd
import sqlite3
from pathlib import Path

# Paths
DATA_RAW       = Path("data/raw")
DATA_PROCESSED = Path("data/processed")
DB_PATH        = Path("njenga.db")

# Remove old DB so we always start clean
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
print(f"Database created: {DB_PATH.resolve()}")


Database created: /home/miringu/Documents/njenga-data/njenga.db


## 1. CAHF — Housing Development Cost Breakdown

**Source:** Centre for Affordable Housing Finance in Africa  
**Report:** Housing Development Cost Benchmarking in Kenya 2022  
**Coverage:** Nairobi only, three typologies, Q3 2022

### Column guide
| Column | Meaning |
|---|---|
| `county` | Always Nairobi — the only city benchmarked |
| `unit_type` | 55m2_house / 2BR_lowrise / 2BR_highrise |
| `unit_size_m2` | Floor area — lets us compare cost per m2 across typologies |
| `cost_category` | What the money pays for — Land, Construction, Finance etc |
| `amount_kes` | Cost per dwelling unit in KES |
| `pct_of_total` | Share of total development cost |
| `year` | 2022 — matches KCHS survey year |
| `source` | Full citation |

### Cleaning tasks
1. Drop `Total development cost` rows — summary rows, not categories
2. Standardise `cost_category` names — strip extra spaces
3. Confirm no nulls in key columns


In [4]:
# Load
cahf = pd.read_csv(DATA_PROCESSED / "cahf_cost_breakdown.csv")
print(f"Loaded: {len(cahf)} rows")
print(cahf.dtypes)
cahf.head(10)


Loaded: 30 rows
county            object
unit_type         object
unit_size_m2       int64
cost_category     object
amount_kes         int64
pct_of_total     float64
year               int64
source            object
dtype: object


,county,unit_type,unit_size_m2,cost_category,amount_kes,pct_of_total,year,source
0,Nairobi,55m2_house,55,Land,378575,9.57,2022,CAHF HDCB Kenya 2022
1,Nairobi,2BR_lowrise,44,Land,135226,4.48,2022,CAHF HDCB Kenya 2022
2,Nairobi,2BR_highrise,44,Land,130617,3.99,2022,CAHF HDCB Kenya 2022
3,Nairobi,55m2_house,55,Infrastructure,966150,24.43,2022,CAHF HDCB Kenya 2022
4,Nairobi,2BR_lowrise,44,Infrastructure,139228,4.62,2022,CAHF HDCB Kenya 2022
5,Nairobi,2BR_highrise,44,Infrastructure,140461,4.30,2022,CAHF HDCB Kenya 2022
6,Nairobi,55m2_house,55,Compliance,57750,1.46,2022,CAHF HDCB Kenya 2022
7,Nairobi,2BR_lowrise,44,Compliance,56514,1.87,2022,CAHF HDCB Kenya 2022
8,Nairobi,2BR_highrise,44,Compliance,56514,1.73,2022,CAHF HDCB Kenya 2022
9,Nairobi,55m2_house,55,Construction,1833013,46.36,2022,CAHF HDCB Kenya 2022


In [5]:
# Task 1: Drop total rows — they are summaries not categories
cahf_clean = cahf[cahf["cost_category"] != "Total development cost"].copy()
print(f"After dropping totals: {len(cahf_clean)} rows")

# Task 2: Standardise cost_category names
cahf_clean["cost_category"] = cahf_clean["cost_category"].str.strip()
print("\nCategories:", cahf_clean["cost_category"].unique())

# Task 3: Check nulls
print("\nNull counts:")
print(cahf_clean.isnull().sum())


After dropping totals: 27 rows

Categories: ['Land' 'Infrastructure' 'Compliance' 'Construction' 'Professional fees'
 'Other development costs' 'Marketing' 'Finance' 'Developer overhead']

Null counts:
county           0
unit_type        0
unit_size_m2     0
cost_category    0
amount_kes       0
pct_of_total     0
year             0
source           0
dtype: int64


In [6]:
# Verify: pre-construction costs for standalone house
standalone = cahf_clean[cahf_clean["unit_type"] == "55m2_house"]
pre_const = standalone[
    standalone["cost_category"].isin(["Land", "Infrastructure", "Compliance"])
]["pct_of_total"].sum()
print(f"Pre-construction (Land + Infra + Compliance) for 55m2 house: {pre_const:.1f}%")

# Verify: construction share for 2BR low-rise
lowrise = cahf_clean[cahf_clean["unit_type"] == "2BR_lowrise"]
construction_pct = lowrise[
    lowrise["cost_category"] == "Construction"
]["pct_of_total"].values[0]
print(f"Construction share for 2BR low-rise: {construction_pct:.1f}%")


Pre-construction (Land + Infra + Compliance) for 55m2 house: 35.5%
Construction share for 2BR low-rise: 56.8%


In [7]:
# Write to database
cahf_clean.to_sql("cahf_cost_breakdown", conn, if_exists="replace", index=False)
print(f"Written to njenga.db: cahf_cost_breakdown ({len(cahf_clean)} rows)")


Written to njenga.db: cahf_cost_breakdown (27 rows)


## 2. KNBS — Construction Input Price Index

**Source:** Kenya National Bureau of Statistics  
**Reports:** Quarterly CIPI releases, Q1 2021 — Q4 2025  
**Base period:** December 2019 = 100

### Column guide
| Column | Meaning |
|---|---|
| `year` | Calendar year |
| `quarter` | Q1 / Q2 / Q3 / Q4 |
| `category` | Materials / Labour / Equipment / Transport and Fuel |
| `product` | Specific input — Cement, Steel, Sand, Mason/foreman |
| `weight` | KNBS assigned importance in overall index (out of 100) |
| `index_value` | Price level — 177 means 77% more expensive than Dec 2019 |

### Cleaning tasks
1. Standardise product names — `Cement /Lime` → `Cement`
2. Flag quarters with incomplete product coverage (< 34 products)
3. Add `period` column — year + quarter as a sortable string
4. Confirm index_value is numeric and in realistic range (50–350)


In [8]:
# Load
knbs = pd.read_csv(DATA_PROCESSED / "knbs_material_index.csv")
print(f"Loaded: {len(knbs)} rows")
print(f"Years: {sorted(knbs['year'].unique())}")
print(f"Products: {knbs['product'].nunique()}")
knbs.head()


Loaded: 580 rows
Years: [2021, 2022, 2023, 2024, 2025]
Products: 34


,year,quarter,category,product,weight,index_value
0,2021,Q1,Equipment,Compressors,3.04,99.45
1,2021,Q1,Equipment,Equipment-Concrete Mixer,5.16,100.00
2,2021,Q1,Equipment,Equipment-Concrete poker / Vibrator,2.36,100.00
3,2021,Q1,Equipment,Equipment-Excavator and Pedestrian Roller,2.64,99.30
4,2021,Q1,Labour,Carpenter/Painter/Welder/Mechanic,3.53,104.54


In [9]:
# Task 1: Standardise product names
name_map = {
    "Cement /Lime":    "Cement",
    "Hard core":       "Hardcore",
    "BRC Mesh and Steel Reinforcement Bars": "Steel and reinforced bars",
    "BRC Mesh & Reinforcement Bars": "Steel and reinforced bars",
    "Equipment-Concrete Mixer":              "Concrete Mixer",
    "Equipment-Concrete poker / Vibrator":   "Concrete Vibrator",
    "Equipment-Excavator and Pedestrian Roller": "Excavator",
    "Compressors":     "Compressor",
    "Plumber/Electrician": "Plumber/Electrician",
    "Machine /plant operators": "Machine operators",
    "Carpenter/Painter/Welder/Mechanic": "Carpenter/Painter/Welder",
    "Mason/foreman":   "Mason/Foreman",
}
knbs["product"] = knbs["product"].replace(name_map)
knbs["product"] = knbs["product"].str.strip()
print(f"Products after standardisation: {knbs['product'].nunique()}")
print(knbs["product"].unique())


Products after standardisation: 34
['Compressor' 'Concrete Mixer' 'Concrete Vibrator' 'Excavator'
 'Carpenter/Painter/Welder' 'Casual' 'Machine operators' 'Mason/Foreman'
 'Plumber/Electrician' 'Watchman' 'Ballast' 'Cement' 'Chip boards and MDF'
 'Damp Proofing and Anti-termite' 'Doors' 'Electrical fittings'
 'Glass and glass putty' 'Hardcore' 'Locks and iron mongery'
 'Metal doors and windows' 'Paints' 'Paving blocks'
 'Quarry products(waste,dust and murram)' 'Roofing materials' 'Sand'
 'Sanitary fittings' 'Steel and reinforced bars' 'Stones' 'Tiles'
 'Timber and Wood' 'Water fittings' 'Water wastes' 'Fuel' 'Transport']


In [10]:
# Task 2: Flag incomplete quarters
coverage = knbs.groupby(["year","quarter"])["product"].count().reset_index()
coverage.columns = ["year","quarter","product_count"]
coverage["complete"] = coverage["product_count"] >= 30
print("Quarter coverage:")
print(coverage.to_string(index=False))


Quarter coverage:
 year quarter  product_count  complete
 2021      Q1             34      True
 2021      Q2             34      True
 2021      Q3             25     False
 2021      Q4             34      True
 2022      Q1             28     False
 2022      Q2             34      True
 2022      Q3             34      True
 2022      Q4             34      True
 2023      Q1             34      True
 2023      Q2             34      True
 2023      Q3             34      True
 2023      Q4             34      True
 2024      Q1             34      True
 2024      Q2             23     False
 2024      Q4             22     False
 2025      Q1             34      True
 2025      Q2             34      True
 2025      Q3              6     False
 2025      Q4             34      True


In [11]:
# Task 3: Add period column for sorting
knbs["period"] = knbs["year"].astype(str) + "-" + knbs["quarter"]
print("Sample periods:", knbs["period"].unique()[:6])

# Task 4: Confirm index_value range
print(f"\nIndex value range: {knbs['index_value'].min():.2f} — {knbs['index_value'].max():.2f}")
outliers = knbs[(knbs["index_value"] < 50) | (knbs["index_value"] > 350)]
print(f"Outliers outside 50-350 range: {len(outliers)}")


Sample periods: ['2021-Q1' '2021-Q2' '2021-Q3' '2021-Q4' '2022-Q1' '2022-Q2']

Index value range: 81.14 — 189.19
Outliers outside 50-350 range: 0


In [12]:
# Write to database
knbs.to_sql("knbs_material_index", conn, if_exists="replace", index=False)
# Also write coverage table
coverage.to_sql("knbs_quarter_coverage", conn, if_exists="replace", index=False)
print(f"Written to njenga.db: knbs_material_index ({len(knbs)} rows)")
print(f"Written to njenga.db: knbs_quarter_coverage ({len(coverage)} rows)")


Written to njenga.db: knbs_material_index (580 rows)
Written to njenga.db: knbs_quarter_coverage (19 rows)


## 3. KCHS — County Household Income

**Source:** Kenya National Bureau of Statistics  
**Report:** The Kenya Poverty Report 2022 (based on KCHS 2022)  
**Coverage:** 26 counties, rural and urban split

### Column guide
| Column | Meaning |
|---|---|
| `county` | County name |
| `residence` | Rural or Urban |
| `mean_monthly_kes` | Mean monthly expenditure per adult equivalent — skewed by top earners |
| `median_monthly_kes` | Median monthly expenditure per adult equivalent — used for affordability |
| `year` | 2022 |
| `source` | Citation |

### Why median not mean
The mean is pulled upward by high earners in the top quintile.
The median represents the *typical* household — half spend more, half spend less.
For an affordability question, median is always the right measure.

### Cleaning tasks
1. Filter to Urban only — James is building in a city
2. Add `hh_monthly_kes` — median × 3.9 (KNBS average household size)
3. Add `hh_annual_kes` — hh_monthly × 12
4. Remove National row from county analysis
5. Drop Rural rows from main table (keep as reference)


In [13]:
# Load
kchs = pd.read_csv(DATA_PROCESSED / "county_income.csv")
print(f"Loaded: {len(kchs)} rows")
print(f"Counties: {kchs['county'].nunique()}")
print(f"Residence types: {kchs['residence'].unique()}")
kchs.head(10)


Loaded: 50 rows
Counties: 26
Residence types: ['Rural' 'Urban']


,county,residence,mean_monthly_kes,median_monthly_kes,year,source
0,Kirinyaga,Rural,7990.0,6657.0,2022,KCHS 2022
1,Kiambu,Rural,7434.0,6942.0,2022,KCHS 2022
2,Embu,Rural,7050.0,6117.0,2022,KCHS 2022
3,Taita/Taveta,Rural,6672.0,5134.0,2022,KCHS 2022
4,Nakuru,Rural,5941.0,5047.0,2022,KCHS 2022
5,Meru,Rural,5931.0,4930.0,2022,KCHS 2022
6,Nyandarua,Rural,5907.0,5300.0,2022,KCHS 2022
7,Nyamira,Rural,5902.0,5132.0,2022,KCHS 2022
8,Migori,Rural,5834.0,5017.0,2022,KCHS 2022
9,Machakos,Rural,5830.0,5004.0,2022,KCHS 2022


In [14]:
# Task 1: Filter to Urban
kchs_urban = kchs[kchs["residence"] == "Urban"].copy()
print(f"Urban rows: {len(kchs_urban)}")

# Task 2 & 3: Add household income columns
KNBS_AVG_HH_SIZE = 3.9  # KNBS average household size
kchs_urban["hh_monthly_kes"] = (kchs_urban["median_monthly_kes"] * KNBS_AVG_HH_SIZE).round(0)
kchs_urban["hh_annual_kes"]  = (kchs_urban["hh_monthly_kes"] * 12).round(0)

# Task 4: Keep National as a row but flag it
kchs_urban["is_national"] = kchs_urban["county"] == "National"

print("\nKey counties:")
key = kchs_urban[kchs_urban["county"].isin(["National","Nairobi","Mombasa","Nakuru","Kiambu"])][
    ["county","median_monthly_kes","hh_monthly_kes","hh_annual_kes"]
]
print(key.to_string(index=False))


Urban rows: 26

Key counties:
  county  median_monthly_kes  hh_monthly_kes  hh_annual_kes
  Kiambu             11682.0         45560.0       546720.0
National              9433.0         36789.0       441468.0
 Nairobi              9433.0         36789.0       441468.0
  Nakuru             11194.0         43657.0       523884.0
 Mombasa             11597.0         45228.0       542736.0


In [15]:
# Write to database — both urban only and full table
kchs_urban.to_sql("county_income_urban", conn, if_exists="replace", index=False)
kchs.to_sql("county_income_all", conn, if_exists="replace", index=False)
print(f"Written to njenga.db: county_income_urban ({len(kchs_urban)} rows)")
print(f"Written to njenga.db: county_income_all ({len(kchs)} rows)")


Written to njenga.db: county_income_urban (26 rows)
Written to njenga.db: county_income_all (50 rows)


## 4. Integrum — Regional Construction Costs

**Source:** Integrum Construction annual cost reports  
**Coverage:** 3 regions, 2021–2025, standard bungalow per m2

### Column guide
| Column | Meaning |
|---|---|
| `year` | 2021–2025 |
| `region` | Nairobi/Mt Kenya, Coast, Western/Nyanza |
| `typology` | Standard Bungalow — most relevant for James |
| `cost_per_m2` | KES per square metre of built area |
| `yoy_change_pct` | Year-on-year percentage change |
| `source` | Integrum annual report citation |

### Cleaning tasks
1. Handle 2025 nulls for Coast and Western/Nyanza — not yet published
2. Add `total_cost_55m2` — cost_per_m2 × 55 (standard bungalow size)
3. Confirm region names are consistent


In [16]:
# Load
integrum = pd.read_csv(DATA_PROCESSED / "integrum_regional_costs.csv")
print(f"Loaded: {len(integrum)} rows")
print(f"Regions: {integrum['region'].unique()}")
print(f"Nulls: {integrum.isnull().sum().to_dict()}")
integrum


Loaded: 13 rows
Regions: ['Coast' 'Nairobi/Mt Kenya' 'Western/Nyanza']
Nulls: {'year': 0, 'region': 0, 'typology': 0, 'cost_per_m2': 0, 'source': 0, 'yoy_change_pct': 3}


,year,region,typology,cost_per_m2,source,yoy_change_pct
0,2021,Coast,Standard Bungalow,35410,Integrum Construction Cost Report 2021,NaN
1,2022,Coast,Standard Bungalow,36250,Integrum Construction Cost Report 2022,2.37
2,2023,Coast,Standard Bungalow,43250,Integrum Construction Cost Report 2023,19.31
3,2024,Coast,Standard Bungalow,51800,Integrum Construction Cost Report 2024,19.77
4,2021,Nairobi/Mt Kenya,Standard Bungalow,33450,Integrum Construction Cost Report 2021,NaN
5,2022,Nairobi/Mt Kenya,Standard Bungalow,34650,Integrum Construction Cost Report 2022,3.59
6,2023,Nairobi/Mt Kenya,Standard Bungalow,41600,Integrum Construction Cost Report 2023,20.06
7,2024,Nairobi/Mt Kenya,Standard Bungalow,48750,Integrum Construction Cost Report 2024,17.19
8,2025,Nairobi/Mt Kenya,Standard Bungalow,54730,Integrum Construction Cost Report 2025,12.27
9,2021,Western/Nyanza,Standard Bungalow,36300,Integrum Construction Cost Report 2021,NaN


In [17]:
# Task 1: Document nulls — do not fill, they are genuinely missing
null_rows = integrum[integrum["cost_per_m2"].isnull()]
if len(null_rows) > 0:
    print(f"Missing data ({len(null_rows)} rows):")
    print(null_rows[["year","region"]].to_string(index=False))
    print("Reason: 2025 Coast and Western/Nyanza not yet published by Integrum")

# Task 2: Add total cost for standard 55m2 bungalow
integrum["total_cost_55m2"] = integrum["cost_per_m2"] * 55

# Task 3: Region names consistent
print(f"\nRegion names: {integrum['region'].unique()}")



Region names: ['Coast' 'Nairobi/Mt Kenya' 'Western/Nyanza']


In [18]:
# Write to database
integrum.to_sql("integrum_regional_costs", conn, if_exists="replace", index=False)
print(f"Written to njenga.db: integrum_regional_costs ({len(integrum)} rows)")


Written to njenga.db: integrum_regional_costs (13 rows)


## 5. Database Verification

Confirm all four tables loaded correctly into `njenga.db`.


In [ ]:
# List all tables
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
print("Tables in njenga.db:")
print(tables.to_string) # use (index=False) to remove the numbers.

# Row counts
for table in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn)["n"].values[0]
    print(f"  {table:35} {count:>5} rows")


Tables in njenga.db:
<bound method DataFrame.to_string of                       name
0      cahf_cost_breakdown
1        county_income_all
2      county_income_urban
3  integrum_regional_costs
4      knbs_material_index
5    knbs_quarter_coverage>
  cahf_cost_breakdown                    27 rows
  county_income_all                      50 rows
  county_income_urban                    26 rows
  integrum_regional_costs                13 rows
  knbs_material_index                   580 rows
  knbs_quarter_coverage                  19 rows


In [ ]:
# Quick sanity check — key numbers should match source documents
checks = {
    "CAHF: 2BR low-rise total cost": (
        "SELECT amount_kes FROM cahf_cost_breakdown "
        "WHERE unit_type='2BR_lowrise' AND cost_category='Construction'",
        1_711_501
    ),
    "KNBS: Steel Q1 2021": (
        "SELECT index_value FROM knbs_material_index "
        "WHERE product LIKE '%Steel%' AND year=2021 AND quarter='Q1'",
        120.78
    ),
    "KCHS: Nairobi median/mo": (
        "SELECT median_monthly_kes FROM county_income_urban "
        "WHERE county='Nairobi'",
        9433.0
    ),
    "Integrum: Nairobi 2024": (
        "SELECT cost_per_m2 FROM integrum_regional_costs "
        "WHERE region='Nairobi/Mt Kenya' AND year=2024",
        48750
    ),
}

print("Sanity checks:")
all_pass = True
for label, (query, expected) in checks.items():
    result = pd.read_sql(query, conn)
    actual = result.iloc[0, 0] if len(result) > 0 else None
    status = "PASS" if actual and abs(float(actual) - float(expected)) < 1 else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {status}  {label}: expected {expected}, got {actual}")

print(f"\n{'All checks passed.' if all_pass else 'Some checks failed — review above.'}")


Sanity checks:
  PASS  CAHF: 2BR low-rise total cost: expected 1711501, got 1711501
  PASS  KNBS: Steel Q1 2021: expected 120.78, got 120.78
  PASS  KCHS: Nairobi median/mo: expected 9433.0, got 9433.0
  PASS  Integrum: Nairobi 2024: expected 48750, got 48750

All checks passed.


In [ ]:
# Close connection
conn.close()
print("njenga.db closed. Notebook complete.")
print("\nNext: 02_analysis.ipynb — four SQL queries, four findings.")


njenga.db closed. Notebook complete.

Next: 02_analysis.ipynb — four SQL queries, four findings.
